 <center><h1>QBUS2820 - Predictive Analytics</h1></center>

# Tutorial 08 Task Solution

## Ridge and Lasso Regression Revision

Your task is to perform ridge and lasso regression in order to predict Salary on the Hitters data.

We will use the sklearn package in order to perform ridge and lasso regression. The main functions in this package that we care about are Ridge(), which can be used to fit ridge regression models, and Lasso() which will fit lasso models. They also have cross-validated counterparts: RidgeCV() and LassoCV() that you will need to perform cross validation.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import scale 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV,LinearRegression 
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

Before proceeding, let's first ensure that the missing values have been removed from the data, as described in the previous lab.

In [5]:
df = pd.read_csv('Hitters.csv').dropna().drop('Player', axis = 1)
df.info()
dummies = pd.get_dummies(df[['League', 'Division', 'NewLeague']])

<class 'pandas.core.frame.DataFrame'>
Index: 263 entries, 1 to 321
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   AtBat      263 non-null    int64  
 1   Hits       263 non-null    int64  
 2   HmRun      263 non-null    int64  
 3   Runs       263 non-null    int64  
 4   RBI        263 non-null    int64  
 5   Walks      263 non-null    int64  
 6   Years      263 non-null    int64  
 7   CAtBat     263 non-null    int64  
 8   CHits      263 non-null    int64  
 9   CHmRun     263 non-null    int64  
 10  CRuns      263 non-null    int64  
 11  CRBI       263 non-null    int64  
 12  CWalks     263 non-null    int64  
 13  League     263 non-null    object 
 14  Division   263 non-null    object 
 15  PutOuts    263 non-null    int64  
 16  Assists    263 non-null    int64  
 17  Errors     263 non-null    int64  
 18  Salary     263 non-null    float64
 19  NewLeague  263 non-null    object 
dtypes: float64(1), 

We will now perform ridge regression and the lasso in order to predict `Salary` on the `Hitters` data. Let's set up our data:

In [7]:
y = df.Salary

# Drop the column with the independent variable (Salary), and columns for which we created dummy variables
X_ = df.drop(['Salary', 'League', 'Division', 'NewLeague'], axis = 1).astype('float64')


# Define the feature set X.
X = pd.concat([X_, dummies[['League_N', 'Division_W', 'NewLeague_N']]], axis = 1)

X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 263 entries, 1 to 321
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   AtBat        263 non-null    float64
 1   Hits         263 non-null    float64
 2   HmRun        263 non-null    float64
 3   Runs         263 non-null    float64
 4   RBI          263 non-null    float64
 5   Walks        263 non-null    float64
 6   Years        263 non-null    float64
 7   CAtBat       263 non-null    float64
 8   CHits        263 non-null    float64
 9   CHmRun       263 non-null    float64
 10  CRuns        263 non-null    float64
 11  CRBI         263 non-null    float64
 12  CWalks       263 non-null    float64
 13  PutOuts      263 non-null    float64
 14  Assists      263 non-null    float64
 15  Errors       263 non-null    float64
 16  League_N     263 non-null    bool   
 17  Division_W   263 non-null    bool   
 18  NewLeague_N  263 non-null    bool   
dtypes: bool(3), f

### Ridge Regression

The Ridge() function has an alpha argument ($\lambda$, but with a different name!) that is used to tune the model. We now split the samples into a training set and a test set in order to estimate the test error of ridge regression and the lasso:

In [9]:
# Split data into training and test sets
X_train, X_test , y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

Use cross-validation to choose the tuning parameter alpha. We can do this using the cross-validated ridge regression function, `RidgeCV()`. By default, the function performs generalized cross-validation (an efficient form of LOOCV), though this can be changed using the argument cv.

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
# We'll generate an array of alpha values ranging from very big to very small
alphas = 10**np.linspace(10,-2,100)*0.5

ridgecv = RidgeCV(alphas=alphas, cv=10, scoring='neg_mean_squared_error')
piperidge = make_pipeline(StandardScaler(), ridgecv)

piperidge.fit(X_train, y_train)
print("Best alpha (z‑score):", piperidge.named_steps['ridgecv'].alpha_)

Best alpha (z‑score): 87.37642000038414


Therefore, we see that the value of alpha that results in the smallest cross-validation error is 87.3764. What is the test MSE associated with this value of alpha?

In [13]:
y_preds = piperidge.predict(X_test)
# RMSE
np.sqrt(mean_squared_error(y_test, y_preds))

349.66039571333636

Finally, we refit our ridge regression model on the full data set, using the value of alpha chosen by cross-validation, and examine the coefficient estimates.

In [15]:
piperidge.fit(X, y)
coefs = piperidge.named_steps['ridgecv'].coef_
pd.Series(coefs, index=X.columns)

AtBat         -223.791446
Hits           239.836468
HmRun            3.426922
Runs            -3.323464
RBI              3.463029
Walks          109.205229
Years          -50.714946
CAtBat        -106.694586
CHits          122.210626
CHmRun          57.057614
CRuns          211.846787
CRBI           118.496695
CWalks        -150.846998
PutOuts         77.758934
Assists         40.202855
Errors         -25.005534
League_N        30.381368
Division_W     -61.607703
NewLeague_N    -13.695967
dtype: float64

As expected, none of the coefficients are exactly zero - ridge regression does not perform variable selection! Note that the numbers you see in `coefs` correspond to standardized predictors, not the raw/original scale of X.

### Lasso

We will now see whether the lasso can yield either a more accurate or a more interpretable model than ridge regression. In order to fit a lasso model and perform CV for parameter tuning, we'll use the `LassoCV()` function:

Notice that in the coefficient plot that depending on the choice of tuning parameter, some of the coefficients are exactly equal to zero. We now perform 10-fold cross-validation to choose the best alpha, refit the model, and compute the associated test error:

In [19]:
lassocv = LassoCV(alphas = alphas, cv = 10, max_iter = 100000)
pipelasso = make_pipeline(StandardScaler(), lassocv)
pipelasso.fit(X_train, y_train)
# Coefficients (on the SCALED/Standardized X scale):
# Some of the coefficients are now reduced to exactly zero
coef_scaled = pipelasso.named_steps["lassocv"].coef_
coefs_scaled_series = pd.Series(coef_scaled, index=X_train.columns)
print("Scaled-X coefficients:")
print(coefs_scaled_series)

Scaled-X coefficients:
AtBat           -0.000000
Hits            73.625718
HmRun            0.000000
Runs             0.000000
RBI              0.000000
Walks           45.463017
Years           -0.000000
CAtBat           0.000000
CHits          121.209633
CHmRun           0.000000
CRuns           20.306563
CRBI            39.428222
CWalks           0.000000
PutOuts         48.767215
Assists          0.000000
Errors          -0.000000
League_N        -0.000000
Division_W     -51.211581
NewLeague_N     -0.000000
dtype: float64


We then refit our lasso regression model on the training data set using predictors with non-zero coefficients, and check its performance on the test data.

In [21]:
# Display the names of predictors with non-zero coefficients
selected_predictors = X_train.columns[pipelasso.named_steps['lassocv'].coef_ != 0]
print("Selected predictors:", list(selected_predictors))

# Refit OLS using only those predictors (on raw, unscaled data)
lr = LinearRegression(fit_intercept=True)
lr.fit(X_train[selected_predictors], y_train)

# Predict and compute RMSE
y_preds = lr.predict(X_test[selected_predictors])
rmse = np.sqrt(mean_squared_error(y_test, y_preds))
print("Post-Lasso OLS RMSE:", rmse)

Selected predictors: ['Hits', 'Walks', 'CHits', 'CRuns', 'CRBI', 'PutOuts', 'Division_W']
Post-Lasso OLS RMSE: 339.2543777354155


Finally, we refit our lasso regression model on the full data set using the same predictors, and examine the coefficient estimates.Because this refit is performed on the unscaled features, the coefficients are on the original (raw) scale of X.

In [23]:
lr.fit(X[selected_predictors],y)
pd.Series(lr.coef_, index = selected_predictors)

Hits            2.044013
Walks           2.282939
CHits          -0.050114
CRuns           0.304801
CRBI            0.453902
PutOuts         0.266212
Division_W   -132.867625
dtype: float64

### 📌 **Extra Question**  
**For Ridge or Lasso coefficients, how do we transform those coefficients back to the original scale of the predictors?**

When we standardize predictors inside a pipeline (using `StandardScaler`), the fitted coefficients are based on the **standardized features** (z-scores).  
To recover the coefficients on the **raw/original scale** of $X$, we use:

\begin{aligned}
\hat{{\beta}}^{(\mathrm{original})}_j &= 
   \frac{\hat{{\beta}}^{(\mathrm{scaled})}_j}{\sigma_j}, 
   && \text{for } j=1,2,\dots,p, \\[6pt]
\hat{{\beta}}^{(\mathrm{original})}_0 &= 
   \hat{\beta}^{(\mathrm{scaled})}_0 
   - \sum_{j=1}^p \frac{\mu_j}{\sigma_j}\,\hat{{\beta}}^{(\mathrm{scaled})}_j.
\end{aligned}

---
**Note:** The handling of dummy (categorical) variables in this solution is provided for **illustrative purposes only**.  
In practice, there are several possible approaches:

- **Do not scale** dummy variables if you wish to retain the direct 0 to 1 interpretation of their coefficients.  
- Use more advanced strategies such as **group penalties** (e.g., group lasso) when dealing with categorical variables that have multiple levels.  

A detailed discussion of these alternatives is left for your further exploration.
